# 3.4 – Spy threshold sensitivity analysis (response to Reviewer 3, comment 4)

This notebook reproduces **only the Spy + Gaussian Naive Bayes labeling step** of `3.1_SemiSupervised_LabelingAndValidation.ipynb` and adds the analyses used in the revision:

1. Reproduction of the reported values (387 spies, μ = 0.61, σ = 0.25, Q1 = 0.47, 16.8% of spies below μ−σ, 9,055 reliable negatives, 3,844 promoted positives).
2. Trade-off between the three candidate thresholds (μ−σ, Q1, P10).
3. Sensitivity to the spy fraction (15%, 20%, 25%) and random seed (10 seeds) → **Table S1**.
4. Characterization of the spies scored near zero and their location in the CPRM/SGB sectors.

It does not retrain the Random Forest and does not change any map or table of the main text. The test/train split follows exactly the original notebook (feature-stratified K-Means, k selected by Silhouette in [5, 10], `random_state=42`).

In [1]:
!pip -q install geopandas

import json, io, zipfile, urllib.request
import numpy as np, pandas as pd, geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

REPO = 'https://raw.githubusercontent.com/npca-ufra/flood-ml-paragominas/main/scripts/files/'
DATA_URL = REPO + 'flood_dataset.csv'
SECTORS_ZIP_URL = REPO + 'risk_sectorization_and_streams.zip'

zsh:1: command not found: pip


## 1. Data and train/test split (identical to the original notebook)

In [2]:
data = pd.read_csv(DATA_URL).rename(columns={
    'ksat': 'condutividade_hidraulica_solo', 'slope': 'declividade',
    'distance': 'distancia', 'elevation': 'elevacao'})
data = data.sample(frac=1, random_state=42).reset_index(drop=True)
FEATURES = ['condutividade_hidraulica_solo', 'declividade', 'distancia', 'elevacao', 'hand', 'twi']

Z = StandardScaler().fit_transform(data[FEATURES])
scores = {}
for k in range(5, 11):
    lab = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init='auto').fit_predict(Z)
    scores[k] = silhouette_score(Z, lab)
k_opt = max(scores, key=scores.get)
print('Silhouette:', {k: round(v, 4) for k, v in scores.items()}, '-> k =', k_opt)

data['feature_clusters'] = KMeans(n_clusters=k_opt, init='k-means++', random_state=42, n_init='auto').fit_predict(Z)
data['key'] = data['classes'].astype(str) + '_' + data['feature_clusters'].astype(str)
vc = data['key'].value_counts(); bad = vc[vc < 2].index
if len(bad):
    data.loc[data['key'].isin(bad), 'key'] = vc[vc >= 2].idxmin()

train, test = train_test_split(data, test_size=0.15, random_state=42, stratify=data['key'])
print('Train:', train['classes'].value_counts().to_dict(), '| Test:', test['classes'].value_counts().to_dict())

X = train[FEATURES]; y = train['classes']
X_pos = X[y == 1]; X_unl = X[y == 0]

Silhouette: {5: np.float64(0.2396), 6: np.float64(0.2489), 7: np.float64(0.2618), 8: np.float64(0.2551), 9: np.float64(0.259), 10: np.float64(0.2525)} -> k = 7
Train: {0: 12899, 1: 1938} | Test: {0: 2277, 1: 342}


## 2. Spy labeling function

In [3]:
def spy_run(spy_fraction=0.2, seed=42):
    """Returns spy probabilities, unlabeled probabilities and the spy index."""
    X_spy, X_pos_train = train_test_split(
        X_pos, test_size=1 - spy_fraction, random_state=seed, stratify=np.ones(len(X_pos)))
    X_mix = pd.concat([X_pos_train, X_unl, X_spy])
    y_mix = np.r_[np.ones(len(X_pos_train)), np.zeros(len(X_unl) + len(X_spy))]
    nb = GaussianNB().fit(X_mix, y_mix)
    return nb.predict_proba(X_spy)[:, 1], nb.predict_proba(X_unl)[:, 1], X_spy.index

def thresholds(p_spy):
    return {'mu-sigma': p_spy.mean() - p_spy.std(),
            'Q1': np.percentile(p_spy, 25),
            'P10': np.percentile(p_spy, 10)}

## 3. Reproduction of the reported run and threshold trade-off

In [4]:
p_spy, p_unl, spy_idx = spy_run(0.2, 42)
print(f'Spies: {len(p_spy)} | mean {p_spy.mean():.3f} | sd {p_spy.std():.3f} | '
      f'median {np.median(p_spy):.3f} | Q1 {np.percentile(p_spy,25):.3f} | max {p_spy.max():.3f}')

rows = []
for name, t in thresholds(p_spy).items():
    rows.append({'Threshold rule': name, 'tau': round(t, 3),
                 'Spies below (%)': round(100 * (p_spy < t).mean(), 1),
                 'Reliable negatives': int((p_unl < t).sum()),
                 'Promoted positives': int((p_unl >= t).sum())})
tradeoff = pd.DataFrame(rows); tradeoff

Spies: 387 | mean 0.610 | sd 0.246 | median 0.694 | Q1 0.474 | max 0.869


,Threshold rule,tau,Spies below (%),Reliable negatives,Promoted positives
0,mu-sigma,0.364,16.8,8429,4470
1,Q1,0.474,25.1,9055,3844
2,P10,0.194,10.1,7518,5381


## 4. Sensitivity to spy fraction and seed → Table S1

In [5]:
runs = []
for frac in [0.15, 0.20, 0.25]:
    for seed in range(10):
        ps, pu, _ = spy_run(frac, seed)
        th = thresholds(ps)
        runs.append({'spy_fraction': frac, 'seed': seed,
                     'triggered': (ps < th['mu-sigma']).mean() > 0.10,
                     'spies_below_mu_sigma': (ps < th['mu-sigma']).mean(),
                     'tau_mu_sigma': th['mu-sigma'], 'tau_Q1': th['Q1'], 'tau_P10': th['P10'],
                     'RN_Q1': (pu < th['Q1']).sum()})
runs = pd.DataFrame(runs)

table_s1 = runs.groupby('spy_fraction').agg(
    runs_triggering=('triggered', 'sum'),
    spies_below_mu_sigma_pct=('spies_below_mu_sigma', lambda s: 100 * s.mean()),
    tau_mu_sigma=('tau_mu_sigma', 'mean'), tau_Q1=('tau_Q1', 'mean'), tau_P10=('tau_P10', 'mean'),
    reliable_negatives_Q1=('RN_Q1', 'mean')).round(3)
display(table_s1)
print('Ranges over 30 runs:')
display(runs[['tau_mu_sigma', 'tau_Q1', 'tau_P10']].agg(['min', 'max']).round(3))
table_s1.to_csv('table_S1_spy_sensitivity.csv'); runs.to_csv('spy_sensitivity_all_runs.csv', index=False)

,runs_triggering,spies_below_mu_sigma_pct,tau_mu_sigma,tau_Q1,tau_P10,reliable_negatives_Q1
spy_fraction,,,,,,
0.15,10,17.759,0.358,0.494,0.145,8981.0
0.20,10,18.243,0.342,0.474,0.122,8971.3
0.25,10,18.306,0.334,0.456,0.130,8989.8


Ranges over 30 runs:


,tau_mu_sigma,tau_Q1,tau_P10
min,0.307,0.426,0.028
max,0.386,0.532,0.219


## 5. Spies scored near zero

In [6]:
spies = train.loc[spy_idx].copy(); spies['p'] = p_spy
low = spies[spies['p'] < 0.01]; other = spies[spies['p'] >= 0.05]
print(f'Spies with p < 0.01: {len(low)} ({100*len(low)/len(spies):.1f}%) | p < 0.05: {(spies.p < 0.05).sum()}')
pd.DataFrame({'near-zero spies (median)': low[FEATURES].median(),
              'other spies (median)': other[FEATURES].median(),
              'unlabeled (median)': X_unl.median()}).round(2)

Spies with p < 0.01: 19 (4.9%) | p < 0.05: 23


,near-zero spies (median),other spies (median),unlabeled (median)
condutividade_hidraulica_solo,11.11,10.65,11.24
declividade,2.63,1.82,1.52
distancia,283.02,63.25,127.28
elevacao,81.24,75.87,78.52
hand,4.97,1.91,4.37
twi,15.93,27.36,27.36


### Location of the near-zero spies in the CPRM/SGB sectors

In [7]:
with urllib.request.urlopen(SECTORS_ZIP_URL) as r:
    zipfile.ZipFile(io.BytesIO(r.read())).extractall('sectors')
sectors = gpd.read_file('sectors/risk_sectorization/risk_sectorization_PGM.shp', encoding='latin1')

spies['geometry'] = [Point(json.loads(g)['coordinates']) for g in spies['.geo']]
joined = gpd.sjoin(gpd.GeoDataFrame(spies, crs=4326), sectors[['num_setor', 'descricao', 'geometry']],
                   how='left', predicate='intersects')
joined['near_zero'] = joined['p'] < 0.01
summary = (joined.groupby('num_setor').agg(spies=('p', 'size'), near_zero=('near_zero', 'sum'))
           .query('near_zero > 0').sort_values('near_zero', ascending=False))
display(summary)
print('Near-zero spies not matched to a polygon:', int(joined.loc[joined.num_setor.isna(), 'near_zero'].sum()))
for s in summary.index:
    print('\n', s, '-', sectors.loc[sectors.num_setor == s, 'descricao'].iloc[0])

,spies,near_zero
num_setor,,
PA_PARAGOM_SR_007_CPRM,22,5
PA_PARAGOM_SR_013_CPRM,3,3
PA_PARAGOM_SR_005_CPRM,138,1
PA_PARAGOM_SR_010_CPRM,22,1
PA_PARAGOM_SR_019_CPRM,4,1


Near-zero spies not matched to a polygon: 8

 PA_PARAGOM_SR_007_CPRM - Setor de risco muito alto a inundação, caracterizado pela ocupação das margens de um afluente do igarapé Paragominas (Figuras 1 e 2). O afluente foi incorporado ao sistema de drenagem pluvial da cidade, com diversos trechos com manilhas subdimension

 PA_PARAGOM_SR_013_CPRM - Transbordamento da drenagem em eventos de chuva intensa, devido a canalização inadequada que diminui a vazão do córrego (1). Chuva forte em 2018 resultou em enxurrada que destruiu uma das casas (2). Moradias e pontos comerciais construídos em alvena

 PA_PARAGOM_SR_005_CPRM - Setor de risco muito alto à inundação, caracterizado pela ocupação da planície de inundação do igarapé Paragominas (Figuras 1, 2, 3 e 4). As casas estão localizadas nas margens do rio e são constantemente afetadas pela elevação natural da co

 PA_PARAGOM_SR_010_CPRM - Ocupação na planície de inundação do rio Uraim (1-4). Área com dificuldade de escoamento da água (5-6). An

## 6. K-Means range for the holdout stratification

Number of positive samples per environmental group for different values of k. As k increases, the positive class is fragmented into more small strata; within k ∈ [5, 10], the Silhouette score selected k = 7.

In [8]:
rows = []
for k in range(3, 16):
    lab = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init='auto').fit_predict(Z)
    pos = pd.Series(lab[data['classes'].values == 1]).value_counts().reindex(range(k), fill_value=0)
    rows.append({'k': k, 'silhouette': round(silhouette_score(Z, lab), 4) if 5 <= k <= 10 else None,
                 'smallest positive group': int(pos.min()),
                 'positive groups < 50 samples': int((pos < 50).sum())})
pd.DataFrame(rows)

,k,silhouette,smallest positive group,positive groups < 50 samples
0,3,NaN,40,1
1,4,NaN,31,1
2,5,0.2396,55,0
3,6,0.2489,2,1
4,7,0.2618,0,2
5,8,0.2551,0,3
6,9,0.2590,0,3
7,10,0.2525,0,4
8,11,NaN,0,4
9,12,NaN,0,4


## 7. Spatial cross-validation: separation between training and validation samples

Reproduces the Spectral Clustering groups (geographic coordinates, nearest-neighbors affinity, 10 groups, `random_state=42`) on the training partition, in the same row order as the original notebook (original positives first, then unlabeled samples), and measures the distance from each validation sample to the nearest training sample under the five spatial folds and under a random five-fold partition.

In [9]:
from sklearn.cluster import SpectralClustering
from sklearn.model_selection import GroupKFold
from scipy.spatial import cKDTree

train_ord = pd.concat([train[train['classes'] == 1], train[train['classes'] == 0]])
lonlat = np.array([json.loads(g)['coordinates'] for g in train_ord['.geo']])
lat0 = np.deg2rad(lonlat[:, 1].mean())
print(f'Mean latitude: {np.rad2deg(lat0):.2f} deg | cos(lat) = {np.cos(lat0):.4f}')
xy_m = np.c_[lonlat[:, 0] * 111320 * np.cos(lat0), lonlat[:, 1] * 110574]   # metres

groups = SpectralClustering(n_clusters=10, random_state=42, assign_labels='discretize',
                            affinity='nearest_neighbors', n_neighbors=10).fit_predict(lonlat)

extent = pd.DataFrame([{'group': g, 'samples': int((groups == g).sum()),
                        'extent_x_km': np.ptp(xy_m[groups == g, 0]) / 1e3,
                        'extent_y_km': np.ptp(xy_m[groups == g, 1]) / 1e3} for g in range(10)]).round(2)
display(extent)

def nn_dist(splits):
    out = []
    for tr_idx, va_idx in splits:
        d, _ = cKDTree(xy_m[tr_idx]).query(xy_m[va_idx]); out.append(d)
    return np.concatenate(out)

d_spatial = nn_dist(GroupKFold(n_splits=5).split(xy_m, groups=groups))
idx = np.random.default_rng(0).permutation(len(xy_m))
d_random = nn_dist([(np.setdiff1d(idx, f), f) for f in np.array_split(idx, 5)])
print(f'Spatial CV : median {np.median(d_spatial):.0f} m | P90 {np.percentile(d_spatial, 90):.0f} m')
print(f'Random CV  : median {np.median(d_random):.0f} m')

Mean latitude: -2.99 deg | cos(lat) = 0.9986


/Users/fabricioaraujo/Library/Python/3.9/lib/python/site-packages/sklearn/manifold/_spectral_embedding.py:329: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


,group,samples,extent_x_km,extent_y_km
0,0,3086,2.38,3.37
1,1,978,0.61,0.55
2,2,1374,0.60,0.85
3,3,1379,0.85,0.66
4,4,1061,0.48,0.70
5,5,1007,0.66,0.61
6,6,1639,0.91,0.80
7,7,1307,0.75,0.60
8,8,1711,3.92,0.74
9,9,1295,0.60,0.89


Spatial CV : median 254 m | P90 599 m
Random CV  : median 15 m


## 8. FSIVI: agreement with the Random Forest and sensitivity to the weights (response to Reviewer 3, comment 5)

Computed at the sample points of the training partition. The FSIVI uses the weights and signs of Eq. 4 of the manuscript, with min–max normalization over the full sample set. The Random Forest probabilities are out-of-fold predictions under the same spatial cross-validation (labels D* from the Spy step with the first-quartile threshold). Results are summarized in **Table S2**.

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import cohen_kappa_score, roc_auc_score
from scipy.stats import spearmanr

# D* labels in the same row order as train_ord (original positives first, then unlabeled)
tau_q1 = np.percentile(p_spy, 25)
new_class = np.r_[np.ones((train['classes'] == 1).sum(), dtype=int), (p_unl >= tau_q1).astype(int)]
Xd = train_ord[FEATURES].reset_index(drop=True)
print('D* composition:', pd.Series(new_class).value_counts().to_dict())

# Out-of-fold Random Forest probabilities under the spatial cross-validation
p_rf = np.zeros(len(Xd))
for tr_idx, va_idx in GroupKFold(n_splits=5).split(Xd, groups=groups):
    rf = RandomForestClassifier(random_state=42).fit(Xd.iloc[tr_idx], new_class[tr_idx])
    p_rf[va_idx] = rf.predict_proba(Xd.iloc[va_idx])[:, 1]
print(f'Check: cross-validated ROC-AUC against D* labels = {roc_auc_score(new_class, p_rf):.3f}')

# FSIVI (Eq. 4): all predictors inversely related to susceptibility except TWI
# Weights = Random Forest importances of the paper (Table 3), read from the repository and normalized to sum to 1
imp = pd.read_csv(REPO + 'feature_importance_flood.csv').rename(columns={
    'ksat': 'condutividade_hidraulica_solo', 'slope': 'declividade',
    'distance': 'distancia', 'elevation': 'elevacao'})[FEATURES].iloc[0]
W = (imp / imp.sum()).to_dict()
print('FSIVI weights:', {k: round(v, 4) for k, v in W.items()})
lo, hi = data[FEATURES].min(), data[FEATURES].max()
def fsivi(weights, X=Xd):
    n = (X - lo) / (hi - lo)
    return sum(weights[c] * (n[c] if c == 'twi' else 1 - n[c]) for c in FEATURES)
f = fsivi(W)

rows = [{'Quantity': 'Spearman correlation, FSIVI vs RF probability', 'Value': spearmanr(f, p_rf)[0]},
        {'Quantity': 'ROC-AUC of FSIVI for RF-predicted flood-prone samples (p >= 0.5)', 'Value': roc_auc_score(p_rf >= 0.5, f)}]
for q, name in [(4, 'quartile'), (5, 'quintile')]:
    cf = pd.qcut(f, q, labels=False).values
    cr = pd.qcut(pd.Series(p_rf).rank(method='first'), q, labels=False).values
    rows += [{'Quantity': f'FSIVI vs RF {name} classes: quadratic-weighted kappa', 'Value': cohen_kappa_score(cf, cr, weights='quadratic')},
             {'Quantity': f'FSIVI vs RF {name} classes: agreement within one class', 'Value': (abs(cf - cr) <= 1).mean()}]

# Sensitivity to the weights
rows.append({'Quantity': 'Spearman correlation, importance-weighted vs equal-weighted FSIVI',
             'Value': spearmanr(f, fsivi({c: 1/6 for c in FEATURES}))[0]})
rng = np.random.default_rng(0); keys = list(W); alpha = np.array([W[k] for k in keys]) * 50
rho = [spearmanr(f, fsivi(dict(zip(keys, rng.dirichlet(alpha)))))[0] for _ in range(1000)]
rows += [{'Quantity': 'Spearman correlation under 1,000 random perturbations of the weights: median', 'Value': np.median(rho)},
         {'Quantity': 'Spearman correlation under 1,000 random perturbations of the weights: minimum', 'Value': np.min(rho)},
         {'Quantity': 'ROC-AUC against the original sector labels (labeled vs unlabeled): FSIVI', 'Value': roc_auc_score(train_ord['classes'], f)},
         {'Quantity': 'ROC-AUC against the original sector labels (labeled vs unlabeled): RF out-of-fold', 'Value': roc_auc_score(train_ord['classes'], p_rf)}]
# Direction check: association of each predictor with the original sector labels vs the direction assigned in Eq. 4
direction = pd.DataFrame({c: {'direction in FSIVI': 'direct' if c == 'twi' else 'inverse',
                              'Spearman with sector label': spearmanr(Xd[c], train_ord['classes'])[0],
                              'Spearman with RF probability': spearmanr(Xd[c], p_rf)[0]} for c in FEATURES}).T
display(direction)
table_s2 = pd.DataFrame(rows).round(3); table_s2.to_csv('table_S2_fsivi_agreement.csv', index=False); table_s2

D* composition: {0: 9055, 1: 5782}


Check: cross-validated ROC-AUC against D* labels = 0.971


FSIVI weights: {'condutividade_hidraulica_solo': 0.2083, 'declividade': 0.1917, 'distancia': 0.1742, 'elevacao': 0.1914, 'hand': 0.1838, 'twi': 0.0507}


,direction in FSIVI,Spearman with sector label,Spearman with RF probability
condutividade_hidraulica_solo,inverse,-0.084627,0.015655
declividade,inverse,0.050836,0.006557
distancia,inverse,-0.188294,-0.541781
elevacao,inverse,-0.263705,-0.742251
hand,inverse,-0.274924,-0.807171
twi,direct,0.235646,0.68527


,Quantity,Value
0,"Spearman correlation, FSIVI vs RF probability",0.793
1,ROC-AUC of FSIVI for RF-predicted flood-prone ...,0.923
2,FSIVI vs RF quartile classes: quadratic-weight...,0.751
3,FSIVI vs RF quartile classes: agreement within...,0.954
4,FSIVI vs RF quintile classes: quadratic-weight...,0.772
5,FSIVI vs RF quintile classes: agreement within...,0.894
6,"Spearman correlation, importance-weighted vs e...",0.981
7,"Spearman correlation under 1,000 random pertur...",0.984
8,"Spearman correlation under 1,000 random pertur...",0.901
9,ROC-AUC against the original sector labels (la...,0.762


## 9. Effect of the threshold rule and spy fraction on the cross-validated ROC-AUC

Supports the statement in Sect. 4.1: substituting µ−σ for the first quartile, or varying the spy fraction between 15% and 25%, changes the positive share of D* by at most about five percentage points and the spatial cross-validation ROC-AUC (Random Forest, mean over the five spatial folds) by less than 0.01.

In [11]:
splits = list(GroupKFold(n_splits=5).split(Xd, groups=groups))
n_pos = int((train['classes'] == 1).sum())

def cv_auc(labels):
    aucs = []
    for tr_idx, va_idx in splits:
        rf = RandomForestClassifier(random_state=42).fit(Xd.iloc[tr_idx], labels[tr_idx])
        aucs.append(roc_auc_score(labels[va_idx], rf.predict_proba(Xd.iloc[va_idx])[:, 1]))
    return np.mean(aucs)

rows = []
for frac in [0.15, 0.20, 0.25]:
    ps, pu, _ = spy_run(frac, 42)
    for rule, tau in [('Q1', np.percentile(ps, 25)), ('mu-sigma', ps.mean() - ps.std())]:
        lab = np.r_[np.ones(n_pos, dtype=int), (pu >= tau).astype(int)]
        rows.append({'spy_fraction': frac, 'rule': rule, 'positive share of D*': lab.mean(), 'CV ROC-AUC': cv_auc(lab)})
auc_sens = pd.DataFrame(rows).round(4)
display(auc_sens)
print('Range of CV ROC-AUC:', auc_sens['CV ROC-AUC'].min(), '-', auc_sens['CV ROC-AUC'].max())

,spy_fraction,rule,positive share of D*,CV ROC-AUC
0,0.15,Q1,0.3923,0.9708
1,0.15,mu-sigma,0.4386,0.9794
2,0.20,Q1,0.3897,0.9710
3,0.20,mu-sigma,0.4319,0.9778
4,0.25,Q1,0.3906,0.9708
5,0.25,mu-sigma,0.4295,0.9795


Range of CV ROC-AUC: 0.9708 - 0.9795
